In [1]:
import yaml
from jinja2 import Template
from langsmith import Client

In [2]:
### RAg Pipeline

def build_prompt(preprocessed_context, question):
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock. 
    
    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Never show the product IDs in the answer.
    - As an output you need to provide:

    * The answer to the question based on the provided context.
    * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
    * Short description (1-2 sentences) of the item based on the description provided in the context.

    - The short description should have the name of the item.
    - The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

    Context:
    {preprocessed_context}

    Question:
    {question}
    """ 
    return prompt


In [3]:
preprocessed_context = "- a \n- b"
question = "What is a?"

In [4]:
prompt = f"""You are a shopping assistant that can answer questions about the products in stock. 
    
    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Never show the product IDs in the answer.
    - As an output you need to provide:

    * The answer to the question based on the provided context.
    * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
    * Short description (1-2 sentences) of the item based on the description provided in the context.

    - The short description should have the name of the item.
    - The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

    Context:
    {preprocessed_context}

    Question:
    {question}
    """ 

### jinja templates

In [10]:
jinja_template = """You are a shopping assistant that can answer questions about the products in stock. 
    
    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Never show the product IDs in the answer.
    - As an output you need to provide:

    * The answer to the question based on the provided context.
    * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
    * Short description (1-2 sentences) of the item based on the description provided in the context.

    - The short description should have the name of the item.
    - The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

    Context:
    {{preprocessed_context}}

    Question:
    {{question}}
    """ 

In [11]:
template=Template(jinja_template)

In [12]:
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [13]:
rendered_template

'You are a shopping assistant that can answer questions about the products in stock. \n\n    You will be given a question and a list of context.\n\n    Instructions:\n    - You need to answer the question based on the provided context only.\n    - Never use word context and refer to it as the available products.\n    - Never show the product IDs in the answer.\n    - As an output you need to provide:\n\n    * The answer to the question based on the provided context.\n    * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.\n    * Short description (1-2 sentences) of the item based on the description provided in the context.\n\n    - The short description should have the name of the item.\n    - The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.\n\n    Context:\n    - a \n- b\n\n    Question:\n    What is a?\n    '

In [15]:
def prompt_template_config(yaml_file,prompt_key):

    with open(yaml_file, 'r') as file:
        config = yaml.safe_load(file)
    template_content=config["prompts"][prompt_key]

    template=Template(template_content)
    return template

In [22]:
def build_prompt_jinja(preprocessed_context, question):

    template=prompt_template_config(yaml_file="prompts/retrieval_generation.yaml", prompt_key="retrieval_generation")
    rendered_prompt = template.render(preprocessed_context=preprocessed_context, question=question)
    return rendered_prompt

In [26]:
print(build_prompt_jinja("hieafddas", "What is the capital of France?"))

You are a shopping assistant that can answer questions about the products in stock. 

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Never show the product IDs in the answer.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
hieafddas

Question:
What is the capital of France?


### prompt registry

In [27]:
ls_client = Client()


In [30]:
ls_template = ls_client.pull_prompt("retrieval-generation")

In [32]:
ls_template.messages[0].prompt.template

'You are a shopping assistant that can answer questions about the products in stock. \n\xa0 \xa0 \n\xa0 \xa0 You will be given a question and a list of context.\n\xa0 \xa0 Instructions:\n\n\xa0 \xa0 - You need to answer the question based on the provided context only.\n\xa0 \xa0 - Never use word context and refer to it as the available products.\n\xa0 \xa0 - Never show the product IDs in the answer.\n\xa0 \xa0 - As an output you need to provide:\n\n\xa0 \xa0 * The answer to the question based on the provided context.\n\xa0 \xa0 * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.\n\xa0 \xa0 * Short description (1-2 sentences) of the item based on the description provided in the context.\n\n\xa0 \xa0 - The short description should have the name of the item.\n\xa0 \xa0 - The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.\n\n\xa0 

In [33]:
def prompt_template_registry( prompt_name):

    template_content=ls_client.pull_prompt(prompt_name).messages[0].prompt.template
    template=Template(template_content)
    return template

In [34]:
print(prompt_template_registry("retrieval-generation").render(preprocessed_context="hieafddas", question="What is the capital of France?"))

You are a shopping assistant that can answer questions about the products in stock. 
    
    You will be given a question and a list of context.
    Instructions:

    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Never show the product IDs in the answer.
    - As an output you need to provide:

    * The answer to the question based on the provided context.
    * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
    * Short description (1-2 sentences) of the item based on the description provided in the context.

    - The short description should have the name of the item.
    - The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

    Context:
    hieafddas

    Question:
    What is the capital of France?
